# Sequence Transformer Monolith Notebook

Bu notebook, `training/mamba2_monolith.py` ve `training/mamba2_monolith.ipynb` tarafindaki **offline traffic-plan learning** fikrini koruyup model katmanini **Sequence Transformer** ile yeniden kurar.

Amac:
- `data/raw/cicids2017/*.csv` kaynaklarini kullanmak
- CSV -> normalize event -> session extraction yapmak
- event-level cok alanli token dizileri uretmek
- autoregressive Sequence Transformer ile next-token objective egitmek
- checkpoint kaydetmek, eval yapmak ve `infer-plan` benzeri session plan uretmek

Onemli notlar:
- Bu notebook **runtime data plane** yerine **offline planner/profile learner** rolundedir.
- Model dogrudan paket veya byte uretmez; **session/event plan** uretir.
- Notebook monolith tutulmustur; tum cekirdek akisi tek dosyada okunabilir olsun diye yardimci modullere dagitilmamistir.


## Notebook Haritasi

Hucreler sirayla su isi yapar:

1. Ortam ve bagimliliklar
2. Veri sozlesmeleri ve plan validasyonu
3. CICIDS2017 CSV -> normalize event extraction
4. Session extraction ve token codec
5. Sequence Transformer modeli
6. Train / eval / checkpoint / infer-plan yuzeyi
7. Ornek config ve adim adim calistirma hucreleri

Bu notebookun tasarim tercihi su: **mimari netlik**, **monolith okunabilirligi** ve **projeye uyum**.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import random
import re
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Literal, Mapping, Optional, Sequence, Tuple

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 1337
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CODE_ROOT = Path.cwd().resolve()
DATA_GLOB = CODE_ROOT / 'data' / 'raw' / 'cicids2017'
OUT_ROOT = CODE_ROOT / 'training' / 'artifacts' / 'sequence_transformer'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'device': str(DEVICE),
    'data_root': str(DATA_GLOB),
    'output_root': str(OUT_ROOT),
}, indent=2, sort_keys=True))

## Core Contracts

Bu hucre, modelin ogrenecegi kavramlari tanimlar.

Buradaki en kritik karar su:
- model ham packet gormez
- model `SessionRecord -> EventRecord[]` sekansini gorur

Boylece Transformer, zaman icinde akan davranis kalibini ogrenmeye odaklanir. Runtime tarafina gecis ise daha sonra `materialize_session_events()` ile olur.

In [ ]:
SUPPORTED_PROTOCOLS = {'http1', 'http2', 'websocket', 'quic'}
SUPPORTED_LABELS = {'benign', 'evasive', 'aggressive'}
SUPPORTED_EVENT_TYPES = {
    'connect_open',
    'tls_client_hello',
    'http_request',
    'http2_headers',
    'ws_open',
    'ws_frame',
    'quic_open',
    'quic_request',
    'idle_wait',
    'retry',
    'reset',
    'close',
}
PROTOCOL_EVENT_TYPES = {
    'http1': {'connect_open', 'tls_client_hello', 'http_request', 'idle_wait', 'retry', 'reset', 'close'},
    'http2': {'connect_open', 'tls_client_hello', 'http2_headers', 'idle_wait', 'retry', 'reset', 'close'},
    'websocket': {'connect_open', 'tls_client_hello', 'ws_open', 'ws_frame', 'idle_wait', 'retry', 'reset', 'close'},
    'quic': {'connect_open', 'quic_open', 'quic_request', 'idle_wait', 'retry', 'reset', 'close'},
}

PAD_TOKEN = '<pad>'
BOS_TOKEN = '<bos>'
EOS_TOKEN = '<eos>'
PROTO_PREFIX = 'PROTO:'
LABEL_PREFIX = 'LABEL:'
EVT_PREFIX = 'EVT:'
DELAY_PREFIX = 'DELAY:'
METHOD_PREFIX = 'METHOD:'
PATH_PREFIX = 'PATH:'
HEADER_PREFIX = 'HEADER:'
PAYLOAD_PREFIX = 'PAYLOAD:'
MUTATION_PREFIX = 'MUTATION:'

KNOWN_DELAY_BUCKETS = ['none', '0_5ms', '6_20ms', '21_100ms', '101_500ms', '500ms_plus']
KNOWN_METHODS = ['none', 'get', 'post', 'put', 'patch', 'delete', 'head', 'options']
KNOWN_PATH_BUCKETS = ['root', 'api', 'ws', 'static', 'auth', 'search', 'media', 'other']
KNOWN_HEADER_PROFILES = ['none', 'unknown', 'modern_browser', 'api_json', 'generic_http', 'http_csv', 'h2_csv', 'quic_csv', 'quic']
KNOWN_PAYLOAD_CLASSES = ['none', 'empty', 'tiny', 'small', 'medium', 'large']
KNOWN_MUTATION_SIGNATURES = [
    'none',
    'csv_attack_label',
    'header_shuffle',
    'retry_burst',
    'fe_balanced',
    'fe_bursty',
    'fe_long_lived',
    'fe_sparse',
    'fe_variable_payload',
    'header_shuffle|retry_burst',
]


@dataclass(frozen=True)
class EventRecord:
    event_type: str
    delay_bucket: str = 'none'
    method: str = 'none'
    path_bucket: str = 'other'
    header_profile: str = 'none'
    payload_class: str = 'none'
    mutation_signature: str = 'none'
    response_time_bucket: str = 'unknown'
    retry_count: int = 0
    close_reason: str = ''


@dataclass(frozen=True)
class SessionRecord:
    schema_version: str
    dataset_origin: str
    trace_id: str
    source_file: str
    split_group_id: str
    session_id: str
    protocol: str
    label: str
    session_metrics: Dict[str, float] = field(default_factory=dict)
    events: List[EventRecord] = field(default_factory=list)


@dataclass(frozen=True)
class SessionEvent:
    step_index: int
    event_type: str
    delay_ms: int = 0
    method: Optional[str] = None
    path: Optional[str] = None
    header_profile: Optional[str] = None
    payload_class: Optional[str] = None
    mutation_flags: List[str] = field(default_factory=list)
    expect_response_class: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class SessionPlan:
    session_id: str
    protocol: Literal['http1', 'http2', 'websocket', 'quic']
    label: Literal['benign', 'evasive', 'aggressive']
    events: List[SessionEvent]

    def to_dict(self) -> Dict[str, Any]:
        return {
            'session_id': self.session_id,
            'protocol': self.protocol,
            'label': self.label,
            'events': [ev.to_dict() for ev in self.events],
        }


@dataclass(frozen=True)
class TrafficPlan:
    plan_id: str
    model_name: str
    checkpoint_id: str
    seed: int
    session_plans: List[SessionPlan]

    def to_dict(self) -> Dict[str, Any]:
        return {
            'plan_id': self.plan_id,
            'model_name': self.model_name,
            'checkpoint_id': self.checkpoint_id,
            'seed': int(self.seed),
            'session_plans': [sp.to_dict() for sp in self.session_plans],
        }


@dataclass
class ValidationResult:
    valid: bool
    errors: List[str] = field(default_factory=list)
    warnings: List[str] = field(default_factory=list)


def validate_plan(plan: TrafficPlan) -> ValidationResult:
    errors: List[str] = []
    warnings: List[str] = []
    if not plan.session_plans:
        errors.append('session_plans must be non-empty')
    for sp in plan.session_plans:
        if sp.protocol not in SUPPORTED_PROTOCOLS:
            errors.append(f'{sp.session_id}: unsupported protocol={sp.protocol}')
        if sp.label not in SUPPORTED_LABELS:
            errors.append(f'{sp.session_id}: unsupported label={sp.label}')
        if not sp.events:
            errors.append(f'{sp.session_id}: empty event sequence')
            continue
        if sp.events[0].event_type != 'connect_open':
            errors.append(f'{sp.session_id}: first event must be connect_open')
        if sp.events[-1].event_type not in {'close', 'reset'}:
            errors.append(f'{sp.session_id}: last event must be close|reset')
        allowed = PROTOCOL_EVENT_TYPES.get(sp.protocol, set())
        for idx, ev in enumerate(sp.events):
            if ev.event_type not in allowed:
                errors.append(f'{sp.session_id}: event_type={ev.event_type} invalid for protocol={sp.protocol}')
            if idx > 0 and ev.event_type == 'connect_open':
                errors.append(f'{sp.session_id}: connect_open can only appear at step 0')
            if idx < len(sp.events) - 1 and ev.event_type in {'close', 'reset'}:
                errors.append(f'{sp.session_id}: terminal event appears early')
    if len(plan.session_plans) < 2:
        warnings.append('very small plan: session_plans < 2')
    return ValidationResult(valid=(len(errors) == 0), errors=errors, warnings=warnings)


print('contracts_loaded')

## CICIDS2017 -> Normalized Event Extraction

Bu hucre notebookun en kritik veri donusum katmanidir.

Burada yaptigimiz sey su:
- her CSV satirini packet gibi gormek yerine **flow-level davranis ozeti** olarak almak
- bu flow bilgisini **runtime'a daha yakin event sequence** haline acmak
- boylece Transformer'a ham tablodan daha anlamli bir sequence vermek

Bu tercih, mevcut Mamba monolith ile ayni felsefiyi korur: model **protocol-valid davranis iskeleti** ogrenir.

In [ ]:
@dataclass(frozen=True)
class NormalizedWireEvent:
    source_file: str
    trace_id: str
    session_id: str
    event_index: int
    timestamp_rel_ms: int
    protocol: str
    event_type: str
    label: str
    src_role: str
    method: str
    path: str
    status_class: str
    payload_class: str
    header_profile: str
    mutation_flags: str
    tls_profile: str
    h2_profile: str
    quic_mode: str
    response_time_bucket: str
    retry_count: int
    close_reason: str

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def _canon(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(name or '').strip().lower())


def _build_colmap(columns: Iterable[str]) -> Dict[str, str]:
    out: Dict[str, str] = {}
    for c in list(columns):
        cc = _canon(c)
        if cc and cc not in out:
            out[cc] = str(c)
    return out


def _first_non_empty(row: dict, colmap: Dict[str, str], keys: List[str], default: str = '') -> str:
    for k in keys:
        real = colmap.get(_canon(k))
        if not real:
            continue
        v = row.get(real)
        s = str(v).strip() if v is not None else ''
        if s:
            return s
    return str(default or '')


def _to_int(v, default: int = 0) -> int:
    try:
        if v is None:
            return int(default)
        s = str(v).strip()
        if not s:
            return int(default)
        return int(float(s))
    except Exception:
        return int(default)


def _to_float(v, default: float = 0.0) -> float:
    try:
        if v is None:
            return float(default)
        s = str(v).strip()
        if not s:
            return float(default)
        return float(s)
    except Exception:
        return float(default)


def _map_label(raw_label: str, default_label: str = 'benign') -> str:
    s = str(raw_label or '').strip().lower()
    if not s:
        return str(default_label or 'benign')
    if s in {'benign', 'normal'} or 'benign' in s:
        return 'benign'
    evasive_keywords = ('slowloris', 'slowhttptest', 'brute', 'infiltration', 'web attack', 'xss', 'sql', 'heartbleed', 'bot')
    if any(k in s for k in evasive_keywords):
        return 'evasive'
    return 'aggressive'


def _protocol_from_row(proto_raw, src_port: int, dst_port: int) -> str:
    p = str(proto_raw or '').strip().lower()
    if not p:
        return 'http2' if (src_port == 443 or dst_port == 443) else 'http1'
    if p.isdigit():
        n = int(p)
        if n == 17:
            return 'quic' if (src_port == 443 or dst_port == 443) else 'http1'
        if n == 6:
            return 'http2' if (src_port == 443 or dst_port == 443) else 'http1'
    if p == 'udp':
        return 'quic' if (src_port == 443 or dst_port == 443) else 'http1'
    if p == 'tcp':
        return 'http2' if (src_port == 443 or dst_port == 443) else 'http1'
    if 'quic' in p:
        return 'quic'
    if 'http2' in p or p == 'h2':
        return 'http2'
    return 'http1'


def _payload_class_bytes(total_bytes: int) -> str:
    n = int(total_bytes or 0)
    if n <= 0:
        return 'empty'
    if n <= 128:
        return 'tiny'
    if n <= 1024:
        return 'small'
    if n <= 8192:
        return 'medium'
    return 'large'


def _trace_id_for_file(csv_path: str) -> str:
    p = Path(csv_path)
    h = hashlib.sha1(str(p.resolve()).encode('utf-8', errors='replace')).hexdigest()[:12]
    return f'{p.stem}-{h}'


def _canonical_session_id(src_ip: str, sport: int, dst_ip: str, dport: int) -> str:
    a = (str(src_ip), int(sport))
    b = (str(dst_ip), int(dport))
    left, right = (a, b) if a <= b else (b, a)
    return f'{left[0]}:{left[1]}<->{right[0]}:{right[1]}'


def iter_csv_normalized_events(csv_path: str, *, default_label: str = 'benign', max_rows: int = 0, chunksize: int = 100000) -> Iterator[NormalizedWireEvent]:
    file_path = str(csv_path or '').strip()
    if not file_path:
        return
    p = Path(file_path)
    if not p.exists():
        return

    trace_id = _trace_id_for_file(file_path)
    session_next_idx: Dict[str, int] = {}
    row_count = 0
    first_ts = None
    reader = pd.read_csv(file_path, low_memory=False, chunksize=max(1, int(chunksize or 100000)))
    for chunk in reader:
        colmap = _build_colmap(list(chunk.columns))
        for row in chunk.to_dict(orient='records'):
            row_count += 1
            if int(max_rows or 0) > 0 and row_count > int(max_rows):
                return

            src_ip = _first_non_empty(row, colmap, ['source ip', 'src ip'], default='0.0.0.0')
            dst_ip = _first_non_empty(row, colmap, ['destination ip', 'dst ip'], default='0.0.0.0')
            src_port = _to_int(_first_non_empty(row, colmap, ['source port', 'src port'], default='0'))
            dst_port = _to_int(_first_non_empty(row, colmap, ['destination port', 'dst port'], default='0'))
            proto_raw = _first_non_empty(row, colmap, ['protocol'], default='')
            protocol = _protocol_from_row(proto_raw, src_port, dst_port)
            row_label = _first_non_empty(row, colmap, ['label', 'attack', 'class'], default=default_label)
            label = _map_label(row_label, default_label=default_label)

            flow_id = _first_non_empty(row, colmap, ['flow id', 'flowid'], default='')
            session_id = f'flow:{flow_id}' if flow_id else _canonical_session_id(src_ip, src_port, dst_ip, dst_port)

            ts_raw = _first_non_empty(row, colmap, ['timestamp', 'time'], default='')
            ts_val = None
            if ts_raw:
                try:
                    ts_obj = pd.to_datetime(ts_raw, errors='coerce')
                    if hasattr(ts_obj, 'value'):
                        ts_val = float(ts_obj.value) / 1_000_000_000.0
                except Exception:
                    ts_val = None
            if ts_val is None:
                ts_val = float(row_count)
            if first_ts is None:
                first_ts = float(ts_val)
            base_rel_ms = int(max(0.0, (float(ts_val) - float(first_ts)) * 1000.0))

            syn_cnt = _to_int(_first_non_empty(row, colmap, ['syn flag cnt', 'synflagcnt'], default='0'))
            ack_cnt = _to_int(_first_non_empty(row, colmap, ['ack flag cnt', 'ackflagcnt'], default='0'))
            psh_cnt = _to_int(_first_non_empty(row, colmap, ['psh flag cnt', 'pshflagcnt'], default='0'))
            rst_cnt = _to_int(_first_non_empty(row, colmap, ['rst flag cnt', 'rstflagcnt'], default='0'))
            fwd_pkts = _to_int(_first_non_empty(row, colmap, ['total fwd packets', 'total forward packets', 'tot fwd pkts'], default='0'))
            bwd_pkts = _to_int(_first_non_empty(row, colmap, ['total backward packets', 'tot bwd pkts'], default='0'))
            fwd_bytes = _to_int(_first_non_empty(row, colmap, ['total length of fwd packets', 'fwd packet length total'], default='0'))
            bwd_bytes = _to_int(_first_non_empty(row, colmap, ['total length of bwd packets', 'bwd packet length total'], default='0'))
            flow_duration_us = _to_float(_first_non_empty(row, colmap, ['flow duration'], default='0'))
            flow_iat_mean_us = _to_float(_first_non_empty(row, colmap, ['flow iat mean'], default='0'))
            flow_iat_max_us = _to_float(_first_non_empty(row, colmap, ['flow iat max'], default='0'))

            total_pkts = max(0, fwd_pkts) + max(0, bwd_pkts)
            total_bytes = max(0, fwd_bytes) + max(0, bwd_bytes)
            payload_class = _payload_class_bytes(total_bytes)
            retry_count = max(0, syn_cnt - 1)
            if retry_count == 0 and rst_cnt > 0:
                retry_count = 1

            if protocol == 'quic':
                req_event_type = 'quic_request'
                header_profile = 'quic_csv'
                tls_profile = 'none'
                h2_profile = 'none'
                quic_mode = 'csv_inferred'
            elif protocol == 'http2':
                req_event_type = 'http2_headers'
                header_profile = 'h2_csv'
                tls_profile = 'csv_inferred'
                h2_profile = 'csv_inferred'
                quic_mode = 'none'
            else:
                req_event_type = 'http_request'
                header_profile = 'http_csv'
                tls_profile = 'none'
                h2_profile = 'none'
                quic_mode = 'none'

            flow_duration_ms = max(0, int(flow_duration_us / 1000.0))
            iat_mean_ms = max(0, int(flow_iat_mean_us / 1000.0))
            iat_max_ms = max(0, int(flow_iat_max_us / 1000.0))
            if payload_class in {'medium', 'large'}:
                header_profile = 'api_json'
            if psh_cnt > 0 and header_profile == 'http_csv':
                header_profile = 'generic_http'

            mutation_flags = 'csv_attack_label' if label != 'benign' else ('retry_burst' if retry_count > 0 else 'fe_balanced')
            response_bucket = 'slow' if flow_duration_ms >= 1500 else ('medium' if flow_duration_ms >= 300 else ('fast' if flow_duration_ms > 0 else 'unknown'))
            req_count = int(round(math.log2(max(2, fwd_pkts + 1))))
            req_count = max(1, min(6, req_count))
            if total_pkts >= 40:
                req_count = min(8, req_count + 1)
            if label == 'aggressive' and total_pkts >= 20:
                req_count = min(10, req_count + 1)

            sequence: List[str] = ['connect_open']
            if protocol == 'quic':
                sequence.append('quic_open')
            elif protocol in {'http1', 'http2'} and (src_port == 443 or dst_port == 443):
                sequence.append('tls_client_hello')
            for _ in range(retry_count):
                sequence.append('retry')
            for _ in range(req_count):
                sequence.append(req_event_type)
            if iat_max_ms >= 200 or iat_mean_ms >= 80 or flow_duration_ms >= 1000:
                sequence.append('idle_wait')
            sequence.append('reset' if rst_cnt > 0 else 'close')

            method = 'get' if ack_cnt >= psh_cnt else 'post'
            if req_event_type == 'http2_headers':
                method = 'get' if label == 'benign' else ('post' if payload_class in {'medium', 'large'} else 'get')
            if req_event_type == 'quic_request':
                method = 'get' if payload_class in {'empty', 'tiny', 'small'} else 'post'

            if label == 'aggressive':
                path = '/api/v1/bulk'
            elif label == 'evasive':
                path = '/search?q=health'
            elif payload_class in {'medium', 'large'}:
                path = '/api/v1/resource'
            else:
                path = '/'

            def path_bucket_for(p: str) -> str:
                if p == '/':
                    return 'root'
                if p.startswith('/api'):
                    return 'api'
                if p.startswith('/ws'):
                    return 'ws'
                if p.startswith('/login'):
                    return 'auth'
                if p.startswith('/search'):
                    return 'search'
                return 'other'

            for ev_type in sequence:
                event_index = int(session_next_idx.get(session_id, 0))
                session_next_idx[session_id] = event_index + 1
                yield NormalizedWireEvent(
                    source_file=str(p.name),
                    trace_id=trace_id,
                    session_id=session_id,
                    event_index=event_index,
                    timestamp_rel_ms=base_rel_ms + event_index * max(1, iat_mean_ms or 1),
                    protocol=protocol,
                    event_type=ev_type,
                    label=label,
                    src_role='client',
                    method=method if ev_type in {'http_request', 'http2_headers', 'quic_request'} else 'none',
                    path=path if ev_type in {'http_request', 'http2_headers', 'quic_request'} else '',
                    status_class='2xx',
                    payload_class=payload_class if ev_type in {'http_request', 'http2_headers', 'quic_request'} else 'none',
                    header_profile=header_profile if ev_type in {'http_request', 'http2_headers', 'quic_request'} else 'none',
                    mutation_flags=mutation_flags,
                    tls_profile=tls_profile,
                    h2_profile=h2_profile,
                    quic_mode=quic_mode,
                    response_time_bucket=response_bucket,
                    retry_count=retry_count,
                    close_reason='reset' if ev_type == 'reset' else ('clean' if ev_type == 'close' else ''),
                )


def bucket_delay(delay_ms: int) -> str:
    n = int(delay_ms or 0)
    if n <= 0:
        return 'none'
    if n <= 5:
        return '0_5ms'
    if n <= 20:
        return '6_20ms'
    if n <= 100:
        return '21_100ms'
    if n <= 500:
        return '101_500ms'
    return '500ms_plus'


def path_to_bucket(path: str) -> str:
    p = str(path or '').strip().lower()
    if p in {'', '/'}:
        return 'root'
    if p.startswith('/api'):
        return 'api'
    if p.startswith('/ws'):
        return 'ws'
    if p.startswith('/static'):
        return 'static'
    if p.startswith('/login'):
        return 'auth'
    if p.startswith('/search'):
        return 'search'
    if p.startswith('/media'):
        return 'media'
    return 'other'


def normalized_events_to_sessions(events: Sequence[NormalizedWireEvent], *, max_sessions: int = 0) -> List[SessionRecord]:
    grouped: Dict[Tuple[str, str, str, str], List[NormalizedWireEvent]] = {}
    for ev in list(events or []):
        key = (str(ev.trace_id), str(ev.session_id), str(ev.protocol), str(ev.label))
        grouped.setdefault(key, []).append(ev)

    sessions: List[SessionRecord] = []
    for (trace_id, session_id, protocol, label), seq in grouped.items():
        seq = sorted(seq, key=lambda x: (int(x.event_index), int(x.timestamp_rel_ms)))
        records: List[EventRecord] = []
        source_file = str(seq[0].source_file or '') if seq else ''
        prev_ts = None
        for ev in seq:
            delay_ms = 0 if prev_ts is None else max(0, int(ev.timestamp_rel_ms) - int(prev_ts))
            prev_ts = int(ev.timestamp_rel_ms)
            records.append(EventRecord(
                event_type=str(ev.event_type),
                delay_bucket=bucket_delay(delay_ms),
                method=str(ev.method or 'none').lower(),
                path_bucket=path_to_bucket(str(ev.path or '')),
                header_profile=str(ev.header_profile or 'none').lower(),
                payload_class=str(ev.payload_class or 'none').lower(),
                mutation_signature=str(ev.mutation_flags or 'none').lower(),
                response_time_bucket=str(ev.response_time_bucket or 'unknown').lower(),
                retry_count=int(ev.retry_count or 0),
                close_reason=str(ev.close_reason or '').lower(),
            ))
        if records:
            split_group_id = f'trace_session:{trace_id}:{session_id}'
            session_metrics = {
                'total_pkts': float(sum(1 for x in seq)),
                'total_bytes': float(sum(0 if str(x.payload_class or 'none') == 'none' else 1 for x in seq)),
                'flow_duration_ms': float(max(0, int(seq[-1].timestamp_rel_ms) - int(seq[0].timestamp_rel_ms))) if seq else 0.0,
                'iat_mean_ms': float(sum(max(0, int(seq[i].timestamp_rel_ms) - int(seq[i-1].timestamp_rel_ms)) for i in range(1, len(seq))) / max(1, len(seq) - 1)) if len(seq) > 1 else 0.0,
                'iat_max_ms': float(max([0] + [max(0, int(seq[i].timestamp_rel_ms) - int(seq[i-1].timestamp_rel_ms)) for i in range(1, len(seq))])),
                'packet_len_std': 0.0,
            }
            sessions.append(SessionRecord(schema_version='samurai_event_v1', dataset_origin='cicids2017', trace_id=trace_id, source_file=source_file, split_group_id=split_group_id, session_id=session_id, protocol=protocol, label=label, session_metrics=session_metrics, events=records))
        if int(max_sessions or 0) > 0 and len(sessions) >= int(max_sessions):
            break
    return sessions


def _file_profiles(files: Sequence[Path], *, max_rows_per_file: int = 0) -> Dict[str, Dict[str, List[str]]]:
    profiles: Dict[str, Dict[str, List[str]]] = {}
    for csv_file in list(files or []):
        local_events = list(iter_csv_normalized_events(str(csv_file), max_rows=int(max_rows_per_file or 0), chunksize=50000))
        profiles[csv_file.name] = {
            'labels': sorted({str(ev.label or 'benign') for ev in local_events}),
            'protocols': sorted({str(ev.protocol or 'http1') for ev in local_events}),
        }
    return profiles


def _coverage_for(names: Sequence[str], profiles: Dict[str, Dict[str, List[str]]], field: str) -> set[str]:
    covered: set[str] = set()
    for name in list(names or []):
        covered.update(profiles.get(name, {}).get(field, []))
    return covered


def choose_auto_split(files: Sequence[Path], *, max_rows_per_file: int = 0, auto_test_keywords: Sequence[str] = ('ddos', 'portscan')) -> Tuple[List[str], List[str], List[str], Dict[str, Dict[str, List[str]]]]:
    profiles = _file_profiles(files, max_rows_per_file=int(max_rows_per_file or 0))
    names = sorted(profiles)
    test_files = sorted([name for name in names if any(key in name.lower() for key in auto_test_keywords)])
    remaining = [name for name in names if name not in test_files]
    if not remaining:
        raise RuntimeError('Auto split icin train/val adayi kalmadi.')
    candidates: List[Tuple[Tuple[int, int, int, int, str], str]] = []
    for name in remaining:
        train_files = [candidate for candidate in remaining if candidate != name]
        if not train_files:
            continue
        train_labels = _coverage_for(train_files, profiles, 'labels')
        target_labels = _coverage_for([name] + test_files, profiles, 'labels')
        train_protocols = _coverage_for(train_files, profiles, 'protocols')
        target_protocols = _coverage_for([name] + test_files, profiles, 'protocols')
        if not target_labels.issubset(train_labels):
            continue
        if not target_protocols.issubset(train_protocols):
            continue
        labels = set(profiles.get(name, {}).get('labels', []))
        score = (int('evasive' in labels), int('benign' in labels), -int('aggressive' in labels), -len(labels), name)
        candidates.append((score, name))
    if not candidates:
        raise RuntimeError('Auto split train coverage koruyarak val secemedi.')
    candidates.sort(reverse=True)
    val_files = [candidates[0][1]]
    train_files = [name for name in remaining if name not in val_files]
    return sorted(train_files), sorted(val_files), sorted(test_files), profiles


def load_sessions_from_cicids2017(*, data_root: Path, csv_glob: str = '*.csv', max_rows_per_file: int = 0, max_sessions: int = 0, selected_files: Optional[Sequence[str]] = None) -> Tuple[List[SessionRecord], Dict[str, Any]]:
    files = sorted(data_root.glob(csv_glob))
    if not files:
        raise RuntimeError(f'No CSV files found under {data_root}')
    if selected_files:
        selected = {str(x) for x in list(selected_files or [])}
        files = [p for p in files if p.name in selected]
    sessions: List[SessionRecord] = []
    events_total = 0
    for csv_file in files:
        local_events = list(iter_csv_normalized_events(str(csv_file), max_rows=int(max_rows_per_file or 0), chunksize=50000))
        events_total += int(len(local_events))
        local_sessions = normalized_events_to_sessions(local_events, max_sessions=0)
        for sess in local_sessions:
            sessions.append(sess)
            if int(max_sessions or 0) > 0 and len(sessions) >= int(max_sessions):
                break
        if int(max_sessions or 0) > 0 and len(sessions) >= int(max_sessions):
            break
    summary = {
        'csv_files': [str(p) for p in files],
        'events_total': int(events_total),
        'sessions_total': int(len(sessions)),
        'protocols': sorted({s.protocol for s in sessions}),
        'labels': sorted({s.label for s in sessions}),
    }
    return sessions, summary


sessions_preview, sessions_summary = load_sessions_from_cicids2017(data_root=DATA_GLOB, max_rows_per_file=5000, max_sessions=256)
print(json.dumps(sessions_summary, indent=2, sort_keys=True))
print('sample_session_len=', len(sessions_preview[0].events) if sessions_preview else 0)

## Token Codec ve Sequence Dataset

Transformer dogrudan `EventRecord` nesnesi egitmez. Bunun yerine her session su sekilde token dizisine donusur:

- `PROTO:http2`
- `LABEL:evasive`
- `EVT:connect_open`
- `DELAY:none`
- `METHOD:none`
- `PATH:other`
- ...

Bu sayede model sadece event tipini degil, event'in baglamsal alanlarini da birlikte ogrenebilir.

In [ ]:
class TokenCodec:
    def __init__(self, vocab: Sequence[str]) -> None:
        self.vocab = list(vocab)
        self.token_to_id = {tok: idx for idx, tok in enumerate(self.vocab)}
        self.id_to_token = {idx: tok for idx, tok in enumerate(self.vocab)}
        self.pad_id = int(self.token_to_id[PAD_TOKEN])
        self.bos_id = int(self.token_to_id[BOS_TOKEN])
        self.eos_id = int(self.token_to_id[EOS_TOKEN])

        self.event_token_ids = [idx for idx, tok in enumerate(self.vocab) if tok.startswith(EVT_PREFIX)]
        self.method_token_ids = [idx for idx, tok in enumerate(self.vocab) if tok.startswith(METHOD_PREFIX)]
        self.path_token_ids = [idx for idx, tok in enumerate(self.vocab) if tok.startswith(PATH_PREFIX)]
        self.header_token_ids = [idx for idx, tok in enumerate(self.vocab) if tok.startswith(HEADER_PREFIX)]
        self.payload_token_ids = [idx for idx, tok in enumerate(self.vocab) if tok.startswith(PAYLOAD_PREFIX)]
        self.mutation_token_ids = [idx for idx, tok in enumerate(self.vocab) if tok.startswith(MUTATION_PREFIX)]
        self.delay_token_ids = [idx for idx, tok in enumerate(self.vocab) if tok.startswith(DELAY_PREFIX)]

    @classmethod
    def build(cls, sessions: Sequence[SessionRecord]) -> 'TokenCodec':
        vocab: List[str] = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN]
        for proto in sorted(SUPPORTED_PROTOCOLS):
            vocab.append(f'{PROTO_PREFIX}{proto}')
        for label in sorted(SUPPORTED_LABELS):
            vocab.append(f'{LABEL_PREFIX}{label}')
        for event_type in sorted(SUPPORTED_EVENT_TYPES):
            vocab.append(f'{EVT_PREFIX}{event_type}')
        for bucket in KNOWN_DELAY_BUCKETS:
            vocab.append(f'{DELAY_PREFIX}{bucket}')
        for method in KNOWN_METHODS:
            vocab.append(f'{METHOD_PREFIX}{method}')
        for path_bucket in KNOWN_PATH_BUCKETS:
            vocab.append(f'{PATH_PREFIX}{path_bucket}')
        for hp in KNOWN_HEADER_PROFILES:
            vocab.append(f'{HEADER_PREFIX}{hp}')
        for pc in KNOWN_PAYLOAD_CLASSES:
            vocab.append(f'{PAYLOAD_PREFIX}{pc}')
        for ms in KNOWN_MUTATION_SIGNATURES:
            vocab.append(f'{MUTATION_PREFIX}{ms}')

        discovered = set(vocab)
        for session in list(sessions or []):
            discovered.add(f'{PROTO_PREFIX}{session.protocol}')
            discovered.add(f'{LABEL_PREFIX}{session.label}')
            for ev in list(session.events or []):
                discovered.add(f'{EVT_PREFIX}{ev.event_type}')
                discovered.add(f'{DELAY_PREFIX}{ev.delay_bucket}')
                discovered.add(f'{METHOD_PREFIX}{ev.method}')
                discovered.add(f'{PATH_PREFIX}{ev.path_bucket}')
                discovered.add(f'{HEADER_PREFIX}{ev.header_profile}')
                discovered.add(f'{PAYLOAD_PREFIX}{ev.payload_class}')
                discovered.add(f'{MUTATION_PREFIX}{ev.mutation_signature}')
        return cls(sorted(discovered, key=lambda x: (0 if x in {PAD_TOKEN, BOS_TOKEN, EOS_TOKEN} else 1, x)))

    def token_id_for(self, token: str) -> int:
        if token not in self.token_to_id:
            raise KeyError(f'unknown token={token}')
        return int(self.token_to_id[token])

    def decode_token(self, idx: int) -> str:
        return str(self.id_to_token[int(idx)])

    def decode_event_token(self, idx: int) -> Optional[str]:
        tok = self.decode_token(idx)
        if tok.startswith(EVT_PREFIX):
            return tok.split(':', 1)[1]
        return None

    def encode_sequence(self, protocol: str, label: str, events: Sequence[EventRecord], max_len: int) -> List[int]:
        ids: List[int] = [self.bos_id, self.token_id_for(f'{PROTO_PREFIX}{protocol}'), self.token_id_for(f'{LABEL_PREFIX}{label}')]
        for ev in list(events or []):
            ids.extend([
                self.token_id_for(f'{EVT_PREFIX}{ev.event_type}'),
                self.token_id_for(f'{DELAY_PREFIX}{ev.delay_bucket}'),
                self.token_id_for(f'{METHOD_PREFIX}{ev.method}'),
                self.token_id_for(f'{PATH_PREFIX}{ev.path_bucket}'),
                self.token_id_for(f'{HEADER_PREFIX}{ev.header_profile}'),
                self.token_id_for(f'{PAYLOAD_PREFIX}{ev.payload_class}'),
                self.token_id_for(f'{MUTATION_PREFIX}{ev.mutation_signature}'),
            ])
            if len(ids) >= max_len - 1:
                break
        ids.append(self.eos_id)
        return ids[:max_len]

    def allowed_event_token_ids(self, protocol: str) -> List[int]:
        allowed = PROTOCOL_EVENT_TYPES.get(protocol, set())
        return [idx for idx in self.event_token_ids if self.decode_event_token(idx) in allowed]


class SequenceDataset(Dataset):
    def __init__(self, sessions: Sequence[SessionRecord], codec: TokenCodec, max_len: int) -> None:
        self.samples: List[List[int]] = []
        self.max_len = int(max_len)
        self.pad_id = int(codec.pad_id)
        for session in list(sessions or []):
            self.samples.append(codec.encode_sequence(session.protocol, session.label, session.events, int(max_len)))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        seq = list(self.samples[int(idx)])
        if len(seq) < self.max_len:
            seq.extend([self.pad_id] * (self.max_len - len(seq)))
        else:
            seq = seq[:self.max_len]
        return {'input_ids': torch.tensor(seq, dtype=torch.long)}


codec_preview = TokenCodec.build(sessions_preview)
print('vocab_size=', len(codec_preview.vocab))
print(codec_preview.vocab[:20])

## Sequence Transformer Modeli

Burada klasik decoder-only Transformer kullaniliyor.

Neden bu secim?
- isimiz autoregressive next-token prediction
- causal mask ile soldan saga event-field sequence ogrenmek istiyoruz
- Mamba monolith'e en dogrudan karsilik gelen Transformer varyanti bu

Model bilesenleri:
- token embedding
- learned positional embedding
- causal Transformer blocks
- final layer norm + vocab projection

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError('d_model must be divisible by n_heads')
        self.n_heads = int(n_heads)
        self.head_dim = int(d_model // n_heads)
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        bsz, seq_len, d_model = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        att = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(float(self.head_dim))
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        probs = torch.softmax(att, dim=-1)
        probs = self.dropout(probs)
        y = torch.matmul(probs, v)
        y = y.transpose(1, 2).contiguous().view(bsz, seq_len, d_model)
        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, d_model: int, mlp_ratio: int, dropout: float) -> None:
        super().__init__()
        hidden = int(d_model * mlp_ratio)
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: int, dropout: float) -> None:
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model=d_model, n_heads=n_heads, dropout=dropout)
        self.ln_2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model=d_model, mlp_ratio=mlp_ratio, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.ff(self.ln_2(x))
        return x


class SequenceTransformerLM(nn.Module):
    def __init__(self, *, vocab_size: int, max_len: int, d_model: int = 256, n_heads: int = 8, n_layers: int = 6, mlp_ratio: int = 4, dropout: float = 0.1) -> None:
        super().__init__()
        self.max_len = int(max_len)
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model=d_model, n_heads=n_heads, mlp_ratio=mlp_ratio, dropout=dropout)
            for _ in range(int(n_layers))
        ])
        self.norm_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        bsz, seq_len = input_ids.shape
        if seq_len > self.max_len:
            raise ValueError(f'seq_len={seq_len} exceeds max_len={self.max_len}')
        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        return self.head(x)


print('transformer_model_ready')

## Train / Eval / Checkpoint / Infer-Plan

Bu hucre notebookun operasyonel cekirdegi.

Neler var:
- train/val split
- epoch loop
- next-token accuracy ve perplexity
- checkpoint save/load
- checkpoint'ten session plan infer etme

Buradaki hedef Mamba monolith ile API paritesi saglamak: `train`, `eval`, `infer-plan` kavramlari burada da var.

In [ ]:
@dataclass
class TrainConfig:
    model_name: str = 'sequence-transformer-cicids2017-v1'
    max_sequence_length: int = 128
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 6
    mlp_ratio: int = 4
    dropout: float = 0.10
    batch_size: int = 32
    epochs: int = 4
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    seed: int = 1337
    max_rows_per_file: int = 50000
    max_sessions: int = 0


def seed_everything(seed: int) -> None:
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def validate_session_schema(sessions: Sequence[SessionRecord]) -> None:
    for s in list(sessions or []):
        if str(s.schema_version or '') != 'samurai_event_v1':
            raise RuntimeError(f'session {s.session_id}: invalid schema_version={s.schema_version}')
        if not str(s.dataset_origin or '').strip():
            raise RuntimeError(f'session {s.session_id}: dataset_origin is required')
        if not str(s.trace_id or '').strip():
            raise RuntimeError(f'session {s.session_id}: trace_id is required')
        if not str(s.source_file or '').strip():
            raise RuntimeError(f'session {s.session_id}: source_file is required')
        if not str(s.split_group_id or '').strip():
            raise RuntimeError(f'session {s.session_id}: split_group_id is required')
        if not list(s.events or []):
            raise RuntimeError(f'session {s.session_id}: events must be non-empty')


def validate_session_coverage(*, train_sessions: Sequence[SessionRecord], val_sessions: Sequence[SessionRecord]) -> None:
    train_labels = {str(s.label or 'benign') for s in list(train_sessions or [])}
    val_labels = {str(s.label or 'benign') for s in list(val_sessions or [])}
    missing_val_labels = sorted(val_labels - train_labels)
    if missing_val_labels:
        raise RuntimeError(f'Validation labels train coverage disinda: {missing_val_labels}')
    train_protocols = {str(s.protocol or 'http1') for s in list(train_sessions or [])}
    val_protocols = {str(s.protocol or 'http1') for s in list(val_sessions or [])}
    missing_val_protocols = sorted(val_protocols - train_protocols)
    if missing_val_protocols:
        raise RuntimeError(f'Validation protocols train coverage disinda: {missing_val_protocols}')


def validate_codec_coverage(*, codec: TokenCodec, val_sessions: Sequence[SessionRecord], max_len: int) -> None:
    for session in list(val_sessions or []):
        try:
            codec.encode_sequence(session.protocol, session.label, session.events, int(max_len))
        except Exception as exc:
            raise RuntimeError(f'Validation session codec coverage failed for {session.session_id}: {exc}') from exc


@torch.no_grad()
def evaluate_model(model: SequenceTransformerLM, loader: DataLoader, *, pad_id: int, device: torch.device) -> Dict[str, float]:
    model.eval()
    ce = nn.CrossEntropyLoss(ignore_index=int(pad_id), reduction='sum')
    total_loss = 0.0
    total_tokens = 0
    total_hit = 0
    total_pred = 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        logits = model(ids)
        target = ids[:, 1:].contiguous()
        pred = logits[:, :-1, :].contiguous()
        loss = ce(pred.view(-1, pred.size(-1)), target.view(-1))
        mask = target.ne(int(pad_id))
        hit = pred.argmax(dim=-1).eq(target) & mask
        total_hit += int(hit.sum().item())
        total_pred += int(mask.sum().item())
        total_tokens += int(mask.sum().item())
        total_loss += float(loss.item())
    avg_nll = total_loss / float(max(1, total_tokens))
    return {
        'next_event_top1_accuracy': float(total_hit) / float(max(1, total_pred)),
        'transition_perplexity': float(math.exp(min(20.0, avg_nll))),
        'transitions_eval': int(total_pred),
    }


def run_epoch(*, model: SequenceTransformerLM, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device, pad_id: int) -> Dict[str, float]:
    model.train()
    ce = nn.CrossEntropyLoss(ignore_index=int(pad_id), reduction='sum')
    total_loss = 0.0
    total_tokens = 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        logits = model(ids)
        target = ids[:, 1:].contiguous()
        pred = logits[:, :-1, :].contiguous()
        loss = ce(pred.view(-1, pred.size(-1)), target.view(-1))
        token_count = int(target.ne(int(pad_id)).sum().item())
        norm_loss = loss / float(max(1, token_count))
        optimizer.zero_grad(set_to_none=True)
        norm_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += float(loss.item())
        total_tokens += token_count
    return {'nll': float(total_loss / float(max(1, total_tokens)))}


def save_checkpoint(path: str, *, model: SequenceTransformerLM, codec: TokenCodec, config: Dict[str, Any], dataset_meta: Dict[str, Any], metrics: Dict[str, Any], optimizer: Optional[torch.optim.Optimizer] = None) -> None:
    payload = {
        'format': 'sequence_transformer_torch_v1',
        'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'model_name': str(config.get('model', {}).get('name', 'sequence-transformer')),
        'model_contract': 'autoregressive_multi_field_sequence',
        'runtime_materialization': 'token_conditioned_v2',
        'config': dict(config or {}),
        'vocab': list(codec.vocab),
        'dataset': dict(dataset_meta or {}),
        'metrics': dict(metrics or {}),
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
    }
    torch.save(payload, str(path))


def load_checkpoint(path: str, *, map_location: str | torch.device = 'cpu') -> Dict[str, Any]:
    ck = torch.load(str(path), map_location=map_location)
    if not isinstance(ck, dict):
        raise RuntimeError('invalid checkpoint payload')
    if str(ck.get('format', '')) != 'sequence_transformer_torch_v1':
        raise RuntimeError('checkpoint is not sequence_transformer_torch_v1')
    return ck


def build_model_and_codec_from_checkpoint(checkpoint_path: str, *, device: str = 'cpu') -> Tuple[SequenceTransformerLM, TokenCodec, Dict[str, Any]]:
    ck = load_checkpoint(checkpoint_path, map_location=device)
    codec = TokenCodec(vocab=list(ck['vocab']))
    model_cfg = dict((ck.get('config', {}) or {}).get('model', {}) or {})
    model = SequenceTransformerLM(
        vocab_size=len(codec.vocab),
        max_len=int(model_cfg.get('max_sequence_length', 128)),
        d_model=int(model_cfg.get('d_model', 256)),
        n_heads=int(model_cfg.get('n_heads', 8)),
        n_layers=int(model_cfg.get('n_layers', 6)),
        mlp_ratio=int(model_cfg.get('mlp_ratio', 4)),
        dropout=float(model_cfg.get('dropout', 0.1)),
    )
    model.load_state_dict(dict(ck.get('state_dict', {}) or {}), strict=True)
    model.to(torch.device(device))
    model.eval()
    return model, codec, ck


def train_sequence_transformer(config: TrainConfig) -> Dict[str, Any]:
    seed_everything(int(config.seed))
    all_files = sorted(DATA_GLOB.glob('*.csv'))
    train_file_names, val_file_names, test_file_names, split_profiles = choose_auto_split(all_files, max_rows_per_file=int(config.max_rows_per_file or 0))
    train_sessions, train_load_summary = load_sessions_from_cicids2017(
        data_root=DATA_GLOB,
        max_rows_per_file=int(config.max_rows_per_file or 0),
        max_sessions=int(config.max_sessions or 0),
        selected_files=train_file_names,
    )
    val_sessions, val_load_summary = load_sessions_from_cicids2017(
        data_root=DATA_GLOB,
        max_rows_per_file=int(config.max_rows_per_file or 0),
        max_sessions=0,
        selected_files=val_file_names,
    )
    validate_session_schema(train_sessions)
    validate_session_schema(val_sessions)
    validate_session_coverage(train_sessions=train_sessions, val_sessions=val_sessions)
    codec = TokenCodec.build(train_sessions)
    validate_codec_coverage(codec=codec, val_sessions=val_sessions, max_len=int(config.max_sequence_length))
    train_ds = SequenceDataset(train_sessions, codec, max_len=int(config.max_sequence_length))
    val_ds = SequenceDataset(val_sessions, codec, max_len=int(config.max_sequence_length))

    train_loader = DataLoader(train_ds, batch_size=int(config.batch_size), shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=int(config.batch_size), shuffle=False)

    model = SequenceTransformerLM(
        vocab_size=len(codec.vocab),
        max_len=int(config.max_sequence_length),
        d_model=int(config.d_model),
        n_heads=int(config.n_heads),
        n_layers=int(config.n_layers),
        mlp_ratio=int(config.mlp_ratio),
        dropout=float(config.dropout),
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config.learning_rate), weight_decay=float(config.weight_decay))

    best_val_ppl = float('inf')
    best_metrics: Dict[str, Any] = {}
    best_state_dict: Optional[Dict[str, Any]] = None
    metrics_history: List[Dict[str, Any]] = []
    for epoch in range(1, int(config.epochs) + 1):
        train_metrics = run_epoch(model=model, loader=train_loader, optimizer=optimizer, device=DEVICE, pad_id=codec.pad_id)
        val_metrics = evaluate_model(model, val_loader, pad_id=codec.pad_id, device=DEVICE)
        row = {
            'epoch': int(epoch),
            'train_nll': float(train_metrics['nll']),
            'val_acc': float(val_metrics['next_event_top1_accuracy']),
            'val_ppl': float(val_metrics['transition_perplexity']),
        }
        metrics_history.append(row)
        print(json.dumps(row, sort_keys=True))
        if float(val_metrics['transition_perplexity']) < best_val_ppl:
            best_val_ppl = float(val_metrics['transition_perplexity'])
            best_metrics = {
                'best_epoch': int(epoch),
                'best_val_acc': float(val_metrics['next_event_top1_accuracy']),
                'best_val_ppl': float(val_metrics['transition_perplexity']),
                'transitions_eval': int(val_metrics['transitions_eval']),
            }
            best_state_dict = copy.deepcopy(model.state_dict())

    if best_state_dict is None:
        best_state_dict = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state_dict, strict=True)

    run_dir = OUT_ROOT / datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%SZ')
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / 'sequence_transformer.ckpt.pt'
    save_checkpoint(
        str(ckpt_path),
        model=model,
        codec=codec,
        config={
            'model': {
                'name': str(config.model_name),
                'max_sequence_length': int(config.max_sequence_length),
                'd_model': int(config.d_model),
                'n_heads': int(config.n_heads),
                'n_layers': int(config.n_layers),
                'mlp_ratio': int(config.mlp_ratio),
                'dropout': float(config.dropout),
            },
            'training': asdict(config),
        },
        dataset_meta={
            'train_load_summary': train_load_summary,
            'val_load_summary': val_load_summary,
            'split_profiles': split_profiles,
            'train_files': train_file_names,
            'val_files': val_file_names,
            'test_files': test_file_names,
            'sessions_train': int(len(train_sessions)),
            'sessions_val': int(len(val_sessions)),
        },
        metrics=best_metrics,
        optimizer=optimizer,
    )
    summary = {
        'status': 'ok',
        'run_dir': str(run_dir),
        'checkpoint_path': str(ckpt_path),
        'checkpoint_sha1': hashlib.sha1(ckpt_path.read_bytes()).hexdigest(),
        'best_metrics': best_metrics,
        'metrics_history': metrics_history,
        'train_sessions': int(len(train_sessions)),
        'val_sessions': int(len(val_sessions)),
        'train_files': train_file_names,
        'val_files': val_file_names,
        'test_files': test_file_names,
        'vocab_size': int(len(codec.vocab)),
    }
    (run_dir / 'train_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
    return summary


def eval_checkpoint_on_csv_dir(*, checkpoint_path: str, eval_csv_dir: str, selected_files: Optional[Sequence[str]] = None, device: str = 'cpu', batch_size: int = 32, max_rows_per_file: int = 0, max_eval_sessions: int = 0) -> Dict[str, Any]:
    ckpt = Path(checkpoint_path)
    if not ckpt.exists():
        return {'checkpoint_exists': False, 'checkpoint': str(ckpt), 'status': 'missing_checkpoint'}
    model, codec, loaded = build_model_and_codec_from_checkpoint(str(ckpt), device=device)
    sessions, load_summary = load_sessions_from_cicids2017(
        data_root=Path(eval_csv_dir),
        max_rows_per_file=int(max_rows_per_file or 0),
        max_sessions=int(max_eval_sessions or 0),
        selected_files=selected_files,
    )
    validate_codec_coverage(codec=codec, val_sessions=sessions, max_len=int((loaded.get('config', {}) or {}).get('model', {}).get('max_sequence_length', 128)))
    ds = SequenceDataset(sessions, codec, max_len=int((loaded.get('config', {}) or {}).get('model', {}).get('max_sequence_length', 128)))
    loader = DataLoader(ds, batch_size=max(1, int(batch_size)), shuffle=False)
    metrics = evaluate_model(model, loader, pad_id=codec.pad_id, device=torch.device(device))
    result = {
        'checkpoint_exists': True,
        'checkpoint': str(ckpt),
        'status': 'ok',
        'model_contract': str(loaded.get('model_contract', 'autoregressive_multi_field_sequence')),
        'runtime_materialization': str(loaded.get('runtime_materialization', 'token_conditioned_v2')),
        'eval_load_summary': load_summary,
        'sessions_eval': int(len(sessions)),
        'transitions_eval': int(metrics['transitions_eval']),
        'next_event_top1_accuracy': float(round(metrics['next_event_top1_accuracy'], 6)),
        'transition_perplexity': float(round(metrics['transition_perplexity'], 6)),
    }
    return result


def _default_event_record(event_type: str) -> EventRecord:
    request_like = event_type in {'http_request', 'http2_headers', 'quic_request'}
    ws_like = event_type in {'ws_open', 'ws_frame'}
    return EventRecord(
        event_type=str(event_type),
        delay_bucket='6_20ms',
        method='get' if request_like else 'none',
        path_bucket='ws' if ws_like else ('api' if request_like else 'other'),
        header_profile='modern_browser' if request_like else 'none',
        payload_class='small' if request_like else 'none',
        mutation_signature='none',
    )


def _allowed_ids_for_event_field(codec: TokenCodec, prefix: str, protocol: str, event_type: str) -> List[int]:
    request_like = event_type in {'http_request', 'http2_headers', 'quic_request'}
    ws_like = event_type in {'ws_open', 'ws_frame'}
    if prefix == DELAY_PREFIX:
        return list(codec.delay_token_ids)
    if prefix == METHOD_PREFIX:
        if request_like:
            return [i for i in codec.method_token_ids if codec.decode_token(i) != f'{METHOD_PREFIX}none']
        return [codec.token_id_for(f'{METHOD_PREFIX}none')]
    if prefix == PATH_PREFIX:
        if request_like:
            return list(codec.path_token_ids)
        if ws_like:
            return [codec.token_id_for(f'{PATH_PREFIX}ws')]
        return [codec.token_id_for(f'{PATH_PREFIX}other')]
    if prefix == HEADER_PREFIX:
        if request_like:
            return list(codec.header_token_ids)
        return [codec.token_id_for(f'{HEADER_PREFIX}none')]
    if prefix == PAYLOAD_PREFIX:
        if request_like:
            return list(codec.payload_token_ids)
        return [codec.token_id_for(f'{PAYLOAD_PREFIX}none')]
    if prefix == MUTATION_PREFIX:
        return list(codec.mutation_token_ids)
    raise KeyError(f'unsupported prefix={prefix}')


def _allowed_next_event_ids(codec: TokenCodec, protocol: str, generated_events: Sequence[EventRecord], max_events: int) -> List[int]:
    allowed_protocol = list(codec.allowed_event_token_ids(protocol))
    decoded_allowed = {idx: codec.decode_event_token(idx) for idx in allowed_protocol}
    count = len(list(generated_events or []))
    if count == 0:
        return [idx for idx, name in decoded_allowed.items() if name == 'connect_open']
    last_name = str(generated_events[-1].event_type)
    if last_name in {'close', 'reset'}:
        return [int(codec.eos_id)]
    if count >= max(1, int(max_events)):
        terminal = [idx for idx, name in decoded_allowed.items() if name in {'close', 'reset'}]
        return terminal + [int(codec.eos_id)]
    body_allowed = []
    for idx, name in decoded_allowed.items():
        if count > 0 and name == 'connect_open':
            continue
        if count < max(2, int(max_events)) - 1 and name in {'close', 'reset'}:
            continue
        body_allowed.append(idx)
    terminal = [idx for idx, name in decoded_allowed.items() if name in {'close', 'reset'}]
    if count >= 2:
        body_allowed.extend(terminal)
    uniq: List[int] = []
    seen: set[int] = set()
    for idx in body_allowed:
        if idx not in seen:
            uniq.append(int(idx))
            seen.add(int(idx))
    return uniq or terminal or [int(codec.eos_id)]


def materialize_session_events(event_types: Sequence[EventRecord], *, protocol: str, label: str, session_len: int, seed: int) -> List[Dict[str, Any]]:
    rng = random.Random(int(seed))
    events: List[Dict[str, Any]] = []
    allowed = PROTOCOL_EVENT_TYPES.get(str(protocol), set())
    seq = [ev for ev in list(event_types or []) if ev.event_type in allowed]
    body = [ev for ev in seq if ev.event_type not in {'connect_open', 'close', 'reset'}]
    max_body_len = max(0, int(session_len) - 2)
    body = body[:max_body_len]
    seq = [_default_event_record('connect_open')] + body
    if len(seq) < int(session_len):
        seq.append(_default_event_record('close'))
    else:
        seq[-1] = _default_event_record('close')
    path_map = {'root': '/', 'api': '/api/v1/resource', 'ws': '/ws', 'static': '/static/app.js', 'auth': '/login', 'search': '/search', 'media': '/media/item', 'other': '/resource'}
    delay_ms_map = {'none': 0, '0_5ms': 5, '6_20ms': 12, '21_100ms': 50, '101_500ms': 200, '500ms_plus': 700}
    for i, ev in enumerate(seq):
        method = None
        path = None
        if ev.event_type in {'http_request', 'http2_headers', 'quic_request'}:
            method = str(ev.method or 'GET').upper()
            path = path_map.get(str(ev.path_bucket or 'other'), '/resource')
        elif ev.event_type in {'ws_open', 'ws_frame'}:
            path = '/ws'
        mutation_flags = [] if str(ev.mutation_signature or 'none') == 'none' else [x for x in str(ev.mutation_signature).split('|') if x]
        if not mutation_flags and label in {'evasive', 'aggressive'} and rng.random() < 0.10:
            mutation_flags.append('header_shuffle')
        events.append({
            'step_index': int(i),
            'event_type': str(ev.event_type),
            'delay_ms': int(delay_ms_map.get(str(ev.delay_bucket or 'none'), 20)),
            'method': method,
            'path': path,
            'header_profile': None if str(ev.header_profile or 'none') == 'none' else str(ev.header_profile),
            'payload_class': None if str(ev.payload_class or 'none') == 'none' else str(ev.payload_class),
            'mutation_flags': mutation_flags,
            'expect_response_class': '2xx',
        })
    return events


@torch.no_grad()
def sample_event_types(model: SequenceTransformerLM, codec: TokenCodec, *, protocol: str, label: str, max_events: int, seed: int, device: str = 'cpu', temperature: float = 0.8, top_k: int = 20) -> List[EventRecord]:
    gen = torch.Generator(device='cpu')
    gen.manual_seed(int(seed))
    ids = codec.encode_sequence(protocol=protocol, label=label, events=[], max_len=3)
    model.eval()
    out: List[EventRecord] = []
    field_order = [DELAY_PREFIX, METHOD_PREFIX, PATH_PREFIX, HEADER_PREFIX, PAYLOAD_PREFIX, MUTATION_PREFIX]

    def sample_allowed(allowed_ids: List[int]) -> int:
        x = torch.tensor([ids], dtype=torch.long, device=torch.device(device))
        logits = model(x)[:, -1, :].squeeze(0)
        mask = torch.full_like(logits, float('-inf'))
        mask[allowed_ids] = 0.0
        masked_logits = logits + mask
        masked_logits = masked_logits / max(1e-4, float(temperature))
        if int(top_k) > 0 and int(top_k) < int(len(allowed_ids)):
            vals, idxs = torch.topk(masked_logits, k=int(top_k))
            probs = torch.softmax(vals, dim=-1)
            pick_local = int(torch.multinomial(probs, 1, generator=gen).item())
            return int(idxs[pick_local].item())
        probs = torch.softmax(masked_logits, dim=-1)
        return int(torch.multinomial(probs, 1, generator=gen).item())

    for _ in range(max(1, int(max_events)) + 2):
        next_allowed = _allowed_next_event_ids(codec, protocol, out, int(max_events))
        next_id = sample_allowed(next_allowed)
        if next_id == int(codec.eos_id):
            break
        ids.append(next_id)
        event_type = codec.decode_event_token(next_id)
        if not event_type:
            break
        event = _default_event_record(event_type)
        for prefix in field_order:
            allowed = _allowed_ids_for_event_field(codec, prefix, protocol, event_type)
            field_id = sample_allowed(allowed)
            ids.append(field_id)
            token = codec.decode_token(field_id)
            value = token.split(':', 1)[1] if ':' in token else ''
            if prefix == DELAY_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=value or 'none', method=event.method, path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == METHOD_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=value or 'none', path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == PATH_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=value or 'other', header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == HEADER_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=event.path_bucket, header_profile=value or 'none', payload_class=event.payload_class, mutation_signature=event.mutation_signature)
            elif prefix == PAYLOAD_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=value or 'none', mutation_signature=event.mutation_signature)
            elif prefix == MUTATION_PREFIX:
                event = EventRecord(event_type=event.event_type, delay_bucket=event.delay_bucket, method=event.method, path_bucket=event.path_bucket, header_profile=event.header_profile, payload_class=event.payload_class, mutation_signature=value or 'none')
        out.append(event)
    return out


def infer_plan_from_checkpoint(*, checkpoint_path: str, protocol: str, label: str, session_count: int, session_len: int, seed: int, device: str = 'cpu', temperature: float = 0.8, top_k: int = 20) -> TrafficPlan:
    model, codec, loaded = build_model_and_codec_from_checkpoint(checkpoint_path, device=device)
    session_plans: List[SessionPlan] = []
    for i in range(int(session_count)):
        evs = sample_event_types(model, codec, protocol=protocol, label=label, max_events=max(1, int(session_len) - 1), seed=int(seed) + i, device=device, temperature=temperature, top_k=top_k)
        materialized = materialize_session_events(evs, protocol=protocol, label=label, session_len=session_len, seed=int(seed) + i)
        session_plans.append(SessionPlan(
            session_id=f'infer-{protocol}-{label}-{i:04d}',
            protocol=str(protocol),
            label=str(label),
            events=[SessionEvent(**ev) for ev in materialized],
        ))
    plan = TrafficPlan(
        plan_id=f'plan-{seed}',
        model_name=str(loaded.get('model_name', 'sequence-transformer')),
        checkpoint_id=hashlib.sha1(Path(checkpoint_path).read_bytes()).hexdigest()[:12],
        seed=int(seed),
        session_plans=session_plans,
    )
    vr = validate_plan(plan)
    if not vr.valid:
        raise RuntimeError(f'Inference plan validation failed: {vr.errors}')
    return plan


def eval_checkpoint_on_saved_test_split(*, checkpoint_path: str, device: str = 'cpu', batch_size: int = 32, max_rows_per_file: int = 0, max_eval_sessions: int = 0) -> Dict[str, Any]:
    loaded = load_checkpoint(checkpoint_path, map_location=device)
    dataset_meta = dict(loaded.get('dataset', {}) or {})
    test_files = list(dataset_meta.get('test_files', []) or [])
    if not test_files:
        raise RuntimeError('Checkpoint dataset metadata icinde test_files yok.')
    return eval_checkpoint_on_csv_dir(
        checkpoint_path=checkpoint_path,
        eval_csv_dir=str(DATA_GLOB),
        selected_files=test_files,
        device=device,
        batch_size=batch_size,
        max_rows_per_file=max_rows_per_file,
        max_eval_sessions=max_eval_sessions,
    )


print('train_eval_infer_surface_ready')

## Notebooku Calistirma Akisi

Asagidaki hucreler egitimi otomatik baslatmaz; adim adim ilerlemeni saglar.

Onerilen sira:
1. Config hucresini calistir
2. `train_sequence_transformer()` ile checkpoint uret
3. Cikan checkpoint path ile `infer_plan_from_checkpoint()` calistir
4. Test split uzerinde ayrik eval calistir
5. JSON plani disari yaz veya mevcut runtime/test harness ile bagla


In [ ]:
cfg = TrainConfig(
    model_name='sequence-transformer-cicids2017-v1',
    max_sequence_length=128,
    d_model=256,
    n_heads=8,
    n_layers=6,
    mlp_ratio=4,
    dropout=0.10,
    batch_size=32,
    epochs=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    seed=1337,
    max_rows_per_file=50000,
    max_sessions=0,
)
cfg

In [ ]:
# Bu hucreyi acarak tam egitimi baslatabilirsin.
# train_summary = train_sequence_transformer(cfg)
# print(json.dumps(train_summary, indent=2, sort_keys=True))

In [ ]:
# Checkpoint icindeki test split metadata'sini kullanarak ayrik test eval.
# checkpoint_path = train_summary['checkpoint_path']
# test_eval = eval_checkpoint_on_saved_test_split(
#     checkpoint_path=checkpoint_path,
#     device=str(DEVICE),
#     batch_size=32,
#     max_rows_per_file=0,
#     max_eval_sessions=0,
# )
# print(json.dumps(test_eval, indent=2, sort_keys=True))

In [ ]:
# Ornek infer-plan hucrsi.
# checkpoint_path = train_summary['checkpoint_path']
# plan = infer_plan_from_checkpoint(
#     checkpoint_path=checkpoint_path,
#     protocol='http2',
#     label='evasive',
#     session_count=3,
#     session_len=8,
#     seed=1337,
#     device=str(DEVICE),
#     temperature=0.8,
#     top_k=20,
# )
# print(json.dumps(plan.to_dict(), indent=2, sort_keys=True))

## Kapanis Notu

Bu notebook, projedeki mevcut AI/planner yonune sadik kalarak Sequence Transformer varyantini monolith formatta sunar.

Yani bu dosya:
- Mamba monolith'in veri contract'ini buyuk olcude korur
- model tarafinda ise state-space yerine Transformer kullanir
- runtime'a dogrudan saldiri motoru olarak degil, offline traffic-plan learner olarak bakar

Bir sonraki dogal adim, bu notebooktan cikan checkpoint ve `TrafficPlan` JSON'unu `ddos_samurai` runtime'ina kontrollu sekilde baglamaktir.